# 03 — Candidate-pool scaling microbenchmark (PyTorch)

**Purpose:** isolate the computational behavior of two MMR implementations as the candidate count grows:

- **Incremental MMR:** update each candidate's maximum redundancy after selecting a document, with approximately \(O(kNd)\) similarity work.
- **Full pairwise-matrix MMR:** materialize the \(N\times N\) document-similarity matrix first, which costs approximately \(O(N^2d)\) compute and \(O(N^2)\) memory.

This notebook uses deterministic synthetic embeddings. It is a **pure algorithmic stress test**, not a retrieval-effectiveness experiment and not evidence that a method improves search quality. It is useful to locate break-even points and validate the implementation before measuring real candidate pools.

In [ ]:
import math
import time
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

SEED = 42
DEVICE = "cpu"  # Recommended for small ranking kernels; optionally try "cuda" if available
EMBEDDING_DIM = 384
TOP_K = 10
LAMBDA_MULT = 0.65
CANDIDATE_COUNTS = [10, 25, 50, 100, 250, 500]
WARMUP_REPEATS = 3
TIMING_REPEATS = 20


def select_mmr_incremental(
    query_embedding: torch.Tensor,
    document_embeddings: torch.Tensor,
    lambda_mult: float = 0.65,
    top_k: int = 10,
    redundancy_floor: float = 0.0,
) -> list[int]:
    """Select top-k documents using incremental redundancy updates."""
    if not 0 <= lambda_mult <= 1:
        raise ValueError("lambda_mult must be between 0 and 1")
    relevance = document_embeddings @ query_embedding
    n_docs = document_embeddings.shape[0]
    available = torch.ones(n_docs, dtype=torch.bool, device=document_embeddings.device)
    max_redundancy = torch.zeros(n_docs, dtype=document_embeddings.dtype,
                                  device=document_embeddings.device)
    selected = []
    for _ in range(min(top_k, n_docs)):
        scores = lambda_mult * relevance - (1.0 - lambda_mult) * max_redundancy
        scores = scores.masked_fill(~available, -torch.inf)
        chosen = int(torch.argmax(scores).item())
        selected.append(chosen)
        available[chosen] = False
        similarity = document_embeddings @ document_embeddings[chosen]
        max_redundancy = torch.maximum(
            max_redundancy, torch.clamp(similarity, min=redundancy_floor)
        )
    return selected


def select_mmr_pairwise_matrix(
    query_embedding: torch.Tensor,
    document_embeddings: torch.Tensor,
    lambda_mult: float = 0.65,
    top_k: int = 10,
    redundancy_floor: float = 0.0,
) -> list[int]:
    """Reference implementation that precomputes all N-by-N cosine similarities."""
    relevance = document_embeddings @ query_embedding
    pairwise = document_embeddings @ document_embeddings.T
    if redundancy_floor is not None:
        pairwise = torch.clamp(pairwise, min=redundancy_floor)
    n_docs = document_embeddings.shape[0]
    available = torch.ones(n_docs, dtype=torch.bool, device=document_embeddings.device)
    max_redundancy = torch.zeros(n_docs, dtype=document_embeddings.dtype,
                                  device=document_embeddings.device)
    selected = []
    for _ in range(min(top_k, n_docs)):
        scores = lambda_mult * relevance - (1.0 - lambda_mult) * max_redundancy
        scores = scores.masked_fill(~available, -torch.inf)
        chosen = int(torch.argmax(scores).item())
        selected.append(chosen)
        available[chosen] = False
        max_redundancy = torch.maximum(max_redundancy, pairwise[:, chosen])
    return selected


def synchronize_if_needed(device: torch.device) -> None:
    """Wait for asynchronous CUDA kernels before recording wall-clock latency."""
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def measure_latency_ms(function, device: torch.device, warmups: int, repeats: int) -> np.ndarray:
    """Return individual wall-clock durations in milliseconds."""
    for _ in range(warmups):
        function()
    synchronize_if_needed(device)
    values = []
    for _ in range(repeats):
        synchronize_if_needed(device)
        start = time.perf_counter_ns()
        function()
        synchronize_if_needed(device)
        values.append((time.perf_counter_ns() - start) / 1_000_000.0)
    return np.asarray(values, dtype=np.float64)


def make_embeddings(n_docs: int, dim: int, device: str, seed: int):
    """Generate reproducible normalized embeddings for algorithm-only testing."""
    generator = torch.Generator(device="cpu").manual_seed(seed + n_docs)
    query = F.normalize(torch.randn(dim, generator=generator), dim=0)
    docs = F.normalize(torch.randn(n_docs, dim, generator=generator), dim=1)
    return query.to(device), docs.to(device)


def summarize_timing(values: np.ndarray) -> dict[str, float]:
    """Summarize latency observations without discarding the raw samples."""
    return {
        "mean_ms": float(np.mean(values)),
        "p50_ms": float(np.percentile(values, 50)),
        "p95_ms": float(np.percentile(values, 95)),
        "p99_ms": float(np.percentile(values, 99)),
        "min_ms": float(np.min(values)),
        "max_ms": float(np.max(values)),
    }


torch.manual_seed(SEED)
torch.set_num_threads(1)
if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("CUDA requested but unavailable; set DEVICE='cpu'.")
print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE.startswith("cuda") else None,
})

## 1. Verify that the two implementations produce identical rankings

The pairwise-matrix implementation is used as an equivalence reference. If rankings differ, do not trust the timing comparison; inspect tie-breaking, the redundancy floor, and candidate ordering first.

In [ ]:
query_check, docs_check = make_embeddings(100, EMBEDDING_DIM, DEVICE, SEED)
order_incremental = select_mmr_incremental(query_check, docs_check, LAMBDA_MULT, TOP_K)
order_pairwise = select_mmr_pairwise_matrix(query_check, docs_check, LAMBDA_MULT, TOP_K)
assert order_incremental == order_pairwise, (
    "The implementations differ; debug correctness before benchmarking."
)
print("PASS: incremental and pairwise-matrix implementations return the same top-k order.")

## 2. Measure candidate-count scaling

Use the same query/document vectors for both methods at each candidate count. Synchronize CUDA during timing, include Python selection overhead, and report p50/p95 rather than only the fastest run. The first run may include runtime/allocator effects; warmups are discarded.

In [ ]:
rows = []
raw_latency = {}
for n_docs in CANDIDATE_COUNTS:
    query_vector, document_vectors = make_embeddings(
        n_docs, EMBEDDING_DIM, DEVICE, SEED
    )
    incremental = lambda: select_mmr_incremental(
        query_vector, document_vectors, LAMBDA_MULT, TOP_K
    )
    pairwise = lambda: select_mmr_pairwise_matrix(
        query_vector, document_vectors, LAMBDA_MULT, TOP_K
    )

    # Correctness check at every N, not only the small example.
    assert incremental() == pairwise(), f"Ranking mismatch at N={n_docs}"

    for name, function in [
        ("incremental", incremental),
        ("full_pairwise_matrix", pairwise),
    ]:
        latencies = measure_latency_ms(
            function, torch.device(DEVICE), WARMUP_REPEATS, TIMING_REPEATS
        )
        raw_latency[(n_docs, name)] = latencies
        rows.append({
            "candidate_count": n_docs,
            "method": name,
            "embedding_dim": EMBEDDING_DIM,
            "top_k": min(TOP_K, n_docs),
            **summarize_timing(latencies),
        })

scaling_results = pd.DataFrame(rows).sort_values(["candidate_count", "method"])
display(scaling_results.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for method_name, group in scaling_results.groupby("method"):
    axes[0].plot(group["candidate_count"], group["p50_ms"], marker="o", label=method_name)
    axes[1].plot(group["candidate_count"], group["p95_ms"], marker="o", label=method_name)
for axis in axes:
    axis.set_xlabel("Candidate count N")
    axis.set_ylabel("Latency (ms)")
    axis.set_yscale("log")
    axis.grid(True, alpha=0.25)
    axis.legend()
axes[0].set_title("Median latency")
axes[1].set_title("95th percentile latency")
fig.suptitle(f"Synthetic MMR implementation stress test — device={DEVICE}")
plt.tight_layout()
plt.show()

In [ ]:
scaling_results.to_csv("mmr_scaling_stress_test.csv", index=False)
print("Saved mmr_scaling_stress_test.csv in the current working directory.")

## 3. Interpret the benchmark

- These are timings of synthetic embedding tensors and two implementations, not retrieval effectiveness on a public search corpus.
- The full-matrix method is a comparison point, not necessarily the best optimized implementation of every diversification algorithm.
- Incremental MMR still performs roughly \(kNd\) similarity work. Real end-to-end latency also includes retrieval, text embedding if not cached, feature extraction, serialization, queuing, and network/service overhead.
- Repeat on your target machine and on a realistic candidate pool; report hardware and thread count. Do not combine local CPU timings with Colab GPU timings in one latency ranking without a carefully controlled protocol.